# Telecommunications Customer Churn | Analytics & Predictive Modeling

**Portfolio project** · Exploratory analysis · Chi-square hypothesis test · Leakage-safe machine learning · Retention recommendations

**Goal:** Understand which customer attributes correlate with churn and build a reproducible classification pipeline for churn-risk estimation. Predictions support review and outreach prioritization, not automatic adverse customer decisions.

**Source design:** labeled records from SQL Server (`dbo.LP2_Telco_churn_first_3000`) and a second CSV; the final CSV is reserved for out-of-sample predictions. No performance metrics are calculated from that unlabeled final file.

> Run cells top-to-bottom. Place CSV files in `data/` and optionally configure SQL Server in `.env`. A local first-part CSV can replace SQL Server for portable execution. See `README.md`.

## 1. Setup and configuration

In [ ]:
from pathlib import Path
import json
import os
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import chi2_contingency
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, average_precision_score,
                             classification_report, f1_score, precision_score,
                             recall_score, roc_auc_score, RocCurveDisplay)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
MODEL_DIR = Path("models")
for folder in (OUTPUT_DIR, MODEL_DIR):
    folder.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", 50)


## 2. Load and combine customer records

Set `FIRST_BATCH_CSV` when you do not have access to the original SQL Server. Database credentials are read from environment variables and are never written to notebook output.

In [ ]:
FIRST_BATCH_CSV = DATA_DIR / "LP2_Telco-churn-first-3000.csv"
SECOND_BATCH_CSV = DATA_DIR / "LP2_Telco-churn-second-2000 (1).csv"
PREDICTION_BATCH_CSV = DATA_DIR / "Telco-churn-last-2000.csv"


def load_first_customer_batch(csv_path: Path) -> pd.DataFrame:
    """Load labeled customers from a local CSV, or from the original SQL table."""
    if csv_path.exists():
        return pd.read_csv(csv_path)

    try:
        from dotenv import load_dotenv
        import pyodbc
    except ImportError as exc:
        raise RuntimeError(
            "First-batch CSV missing. Install python-dotenv and pyodbc for SQL Server, "
            "or place the first-batch CSV in data/."
        ) from exc

    load_dotenv()
    server = os.getenv("server_name") or os.getenv("SQL_SERVER")
    database = os.getenv("database_name") or os.getenv("SQL_DATABASE")
    username = os.getenv("user") or os.getenv("SQL_USERNAME")
    password = os.getenv("password") or os.getenv("SQL_PASSWORD")
    driver = os.getenv("SQL_DRIVER", "ODBC Driver 18 for SQL Server")
    if not all((server, database, username, password)):
        raise RuntimeError("SQL Server credentials are missing. See .env.example.")

    connection_string = (
        f"DRIVER={{{driver}}};SERVER={server};DATABASE={database};"
        f"UID={username};PWD={password};Encrypt=yes;TrustServerCertificate=no"
    )
    # Read-only fixed query; close the connection even if the query fails.
    with pyodbc.connect(connection_string) as connection:
        return pd.read_sql_query("SELECT * FROM dbo.LP2_Telco_churn_first_3000", connection)


def load_customer_csv(csv_path: Path) -> pd.DataFrame:
    """Load a project CSV and provide a helpful error if it is unavailable."""
    if not csv_path.is_file():
        raise FileNotFoundError(f"Required file not found: {csv_path.resolve()}")
    return pd.read_csv(csv_path)


first_customer_batch = load_first_customer_batch(FIRST_BATCH_CSV)
second_customer_batch = load_customer_csv(SECOND_BATCH_CSV)
prediction_customers_raw = load_customer_csv(PREDICTION_BATCH_CSV)

# Combine only labeled historical records; reserve the final batch for scoring.
customer_churn_raw = pd.concat(
    [first_customer_batch, second_customer_batch], ignore_index=True
)
print("Training records:", customer_churn_raw.shape)
print("Prediction records:", prediction_customers_raw.shape)
display(customer_churn_raw.head())


## 3. Data validation and preparation

In [ ]:
TARGET_COLUMN = "Churn"
CUSTOMER_ID_COLUMN = "customerID"
NUMERIC_COLUMNS = ["tenure", "MonthlyCharges", "TotalCharges"]
YES_NO_COLUMNS = [
    "Partner", "Dependents", "PhoneService", "MultipleLines",
    "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport",
    "StreamingTV", "StreamingMovies", "PaperlessBilling", "Churn",
]


def standardize_customer_records(customer_data: pd.DataFrame) -> pd.DataFrame:
    """Normalize field types without learning statistics from the dataset."""
    clean_customers = customer_data.copy()
    clean_customers.columns = clean_customers.columns.str.strip()

    for column in NUMERIC_COLUMNS:
        if column in clean_customers:
            clean_customers[column] = pd.to_numeric(clean_customers[column], errors="coerce")

    if "SeniorCitizen" in clean_customers:
        clean_customers["SeniorCitizen"] = (
            clean_customers["SeniorCitizen"]
            .replace({1: "Yes", 0: "No", "1": "Yes", "0": "No", True: "Yes", False: "No"})
            .astype("string")
        )

    for column in YES_NO_COLUMNS:
        if column in clean_customers:
            # Keep missing and service-specific categories (e.g. 'No internet service').
            clean_customers[column] = clean_customers[column].replace(
                {True: "Yes", False: "No", "True": "Yes", "False": "No"}
            )
    return clean_customers


customer_churn_data = standardize_customer_records(customer_churn_raw)
prediction_customers = standardize_customer_records(prediction_customers_raw)

if TARGET_COLUMN not in customer_churn_data:
    raise ValueError("Training records must include a 'Churn' target column.")

# A missing label cannot be used for supervised model training.
customer_churn_data = customer_churn_data.dropna(subset=[TARGET_COLUMN]).copy()
customer_churn_data[TARGET_COLUMN] = (
    customer_churn_data[TARGET_COLUMN].astype(str).str.strip().str.title()
)
if not set(customer_churn_data[TARGET_COLUMN].unique()).issubset({"Yes", "No"}):
    raise ValueError("Churn must contain only 'Yes' and 'No' labels.")

# Prefer customer identifiers for deduplication; otherwise remove exact duplicates.
if CUSTOMER_ID_COLUMN in customer_churn_data:
    customer_churn_data = customer_churn_data.drop_duplicates(subset=CUSTOMER_ID_COLUMN)
else:
    customer_churn_data = customer_churn_data.drop_duplicates()

print("Prepared labeled records:", len(customer_churn_data))
display(customer_churn_data.isna().sum().sort_values(ascending=False).head(12).to_frame("missing_values"))
display(customer_churn_data.describe(include="all").T.head(20))


## 4. Exploratory data analysis

In [ ]:
customer_churn_data["churn_flag"] = customer_churn_data[TARGET_COLUMN].eq("Yes").astype(int)
churn_rate = customer_churn_data["churn_flag"].mean()
print(f"Observed customer churn rate: {churn_rate:.1%}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=customer_churn_data, x=TARGET_COLUMN, order=["No", "Yes"], ax=axes[0])
axes[0].set(title="Customer Churn Distribution", xlabel="Churn Status", ylabel="Customers")
sns.histplot(data=customer_churn_data, x="tenure", hue=TARGET_COLUMN,
             bins=25, stat="density", common_norm=False, ax=axes[1])
axes[1].set(title="Customer Tenure by Churn", xlabel="Tenure (months)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "churn_distribution.png", dpi=160, bbox_inches="tight")
plt.show()


In [ ]:
def plot_churn_rate_by_segment(
    customer_data: pd.DataFrame, segment_column: str, axis=None
):
    """Plot segment-level churn rates rather than potentially misleading raw counts."""
    segment_rates = (
        customer_data.groupby(segment_column, dropna=False)["churn_flag"]
        .agg(churn_rate="mean", customers="size")
        .sort_values("churn_rate", ascending=False)
        .reset_index()
    )
    if axis is None:
        _, axis = plt.subplots(figsize=(9, 4))
    sns.barplot(data=segment_rates, x=segment_column, y="churn_rate", ax=axis)
    axis.set(title=f"Churn Rate by {segment_column}", ylabel="Churn Rate", xlabel=segment_column)
    axis.tick_params(axis="x", rotation=25)
    axis.set_ylim(0, 1)
    return segment_rates


for segment_name in ["gender", "SeniorCitizen", "Contract", "InternetService", "PaymentMethod"]:
    if segment_name in customer_churn_data:
        segment_summary = plot_churn_rate_by_segment(customer_churn_data, segment_name)
        plt.tight_layout()
        plt.show()
        display(segment_summary)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for axis, column in zip(axes, ["MonthlyCharges", "TotalCharges"]):
    sns.boxplot(data=customer_churn_data, x=TARGET_COLUMN, y=column,
                order=["No", "Yes"], ax=axis)
    axis.set_title(f"{column} by Churn Status")
plt.tight_layout()
plt.show()

# Inspect numeric relationships; correlation does not establish causation.
correlation_columns = [column for column in NUMERIC_COLUMNS if column in customer_churn_data]
correlation_columns.append("churn_flag")
sns.heatmap(customer_churn_data[correlation_columns].corr(), annot=True, cmap="coolwarm", center=0)
plt.title("Numeric Feature Correlations")
plt.tight_layout()
plt.show()


## 5. Hypothesis test: internet service vs. churn

**H₀:** Internet service type and churn status are independent. **H₁:** They are associated. A chi-square test evaluates association, not causality.

In [ ]:
internet_churn_counts = pd.crosstab(
    customer_churn_data["InternetService"], customer_churn_data[TARGET_COLUMN]
)
chi_square_statistic, chi_square_p_value, degrees_of_freedom, expected_counts = (
    chi2_contingency(internet_churn_counts)
)
print(f"Chi-square statistic: {chi_square_statistic:.3f}")
print(f"Degrees of freedom: {degrees_of_freedom}")
print(f"p-value: {chi_square_p_value:.6g}")
print("Decision (alpha=0.05):", "Reject H0" if chi_square_p_value < 0.05 else "Fail to reject H0")
if (expected_counts < 5).any():
    warnings.warn("Some expected counts are below 5; interpret chi-square approximation cautiously.")
display(internet_churn_counts)


## 6. Leakage-safe feature engineering and holdout split

All imputation, encoding, and scaling are fitted on training folds only. The holdout test set is not used for model selection or tuning.

In [ ]:
customer_features = customer_churn_data.drop(columns=[TARGET_COLUMN, "churn_flag", CUSTOMER_ID_COLUMN], errors="ignore")
churn_target = customer_churn_data[TARGET_COLUMN].map({"No": 0, "Yes": 1}).astype(int)

# The prediction file may contain a blank or pre-existing Churn column; never use it as a feature.
prediction_features = prediction_customers.drop(columns=[TARGET_COLUMN, CUSTOMER_ID_COLUMN], errors="ignore")

# Ensure training and scoring datasets have identical feature schemas.
missing_prediction_columns = set(customer_features.columns) - set(prediction_features.columns)
if missing_prediction_columns:
    raise ValueError(f"Prediction dataset lacks training features: {sorted(missing_prediction_columns)}")
prediction_features = prediction_features.reindex(columns=customer_features.columns)

X_train, X_test, y_train, y_test = train_test_split(
    customer_features, churn_target, test_size=0.20,
    random_state=RANDOM_STATE, stratify=churn_target
)

numerical_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.columns.difference(numerical_features).tolist()

numerical_preprocessing = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_preprocessing = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

customer_preprocessor = ColumnTransformer([
    ("numeric", numerical_preprocessing, numerical_features),
    ("categorical", categorical_preprocessing, categorical_features),
])
print(f"Training: {X_train.shape}; holdout: {X_test.shape}")


## 7. Compare classification models using stratified cross-validation

Use **ROC-AUC** for ranking and **average precision** to monitor positive-class detection under imbalance. Cross-validation is performed only on the training partition.

In [ ]:
candidate_churn_models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, min_samples_leaf=2,
                                              class_weight="balanced", n_jobs=-1,
                                              random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
}

stratified_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
model_cv_results = []
for model_name, classifier in candidate_churn_models.items():
    churn_pipeline = Pipeline([
        ("preprocessing", customer_preprocessor),
        ("classifier", classifier),
    ])
    fold_scores = cross_validate(
        churn_pipeline, X_train, y_train, cv=stratified_cv,
        scoring={"roc_auc": "roc_auc", "average_precision": "average_precision", "f1": "f1"},
        n_jobs=1, error_score="raise",
    )
    model_cv_results.append({
        "model": model_name,
        "cv_roc_auc_mean": fold_scores["test_roc_auc"].mean(),
        "cv_roc_auc_std": fold_scores["test_roc_auc"].std(),
        "cv_average_precision": fold_scores["test_average_precision"].mean(),
        "cv_f1": fold_scores["test_f1"].mean(),
    })

churn_model_comparison = pd.DataFrame(model_cv_results).sort_values(
    "cv_roc_auc_mean", ascending=False
)
display(churn_model_comparison)
churn_model_comparison.to_csv(OUTPUT_DIR / "model_cv_comparison.csv", index=False)


## 8. Hyperparameter tuning

Tune the strongest baseline using ROC-AUC and stratified CV. The model remains a single end-to-end pipeline for safe inference.

In [ ]:
selected_model_name = churn_model_comparison.iloc[0]["model"]
selected_classifier = candidate_churn_models[selected_model_name]

search_parameters = {
    "Logistic Regression": {"classifier__C": [0.1, 1.0, 10.0]},
    "Random Forest": {"classifier__max_depth": [None, 10, 20],
                      "classifier__min_samples_leaf": [1, 2, 5]},
    "Gradient Boosting": {"classifier__learning_rate": [0.05, 0.1],
                          "classifier__n_estimators": [100, 200]},
}

churn_model_search = GridSearchCV(
    estimator=Pipeline([
        ("preprocessing", customer_preprocessor),
        ("classifier", selected_classifier),
    ]),
    param_grid=search_parameters[selected_model_name],
    scoring="roc_auc", cv=stratified_cv, n_jobs=-1, refit=True,
)
churn_model_search.fit(X_train, y_train)
best_churn_pipeline = churn_model_search.best_estimator_
print("Selected model:", selected_model_name)
print("Best parameters:", churn_model_search.best_params_)
print(f"Best cross-validation ROC-AUC: {churn_model_search.best_score_:.4f}")


## 9. Final holdout evaluation

Evaluate the tuned model once on the untouched holdout. Threshold 0.50 is illustrative; operational thresholds require a separate cost-benefit and calibration analysis.

In [ ]:
holdout_churn_probability = best_churn_pipeline.predict_proba(X_test)[:, 1]
holdout_churn_prediction = (holdout_churn_probability >= 0.50).astype(int)

holdout_metrics = {
    "model": selected_model_name,
    "roc_auc": roc_auc_score(y_test, holdout_churn_probability),
    "average_precision": average_precision_score(y_test, holdout_churn_probability),
    "accuracy": accuracy_score(y_test, holdout_churn_prediction),
    "precision": precision_score(y_test, holdout_churn_prediction, zero_division=0),
    "recall": recall_score(y_test, holdout_churn_prediction, zero_division=0),
    "f1": f1_score(y_test, holdout_churn_prediction, zero_division=0),
}
display(pd.DataFrame([holdout_metrics]))
print(classification_report(y_test, holdout_churn_prediction,
                            target_names=["Retained", "Churned"], zero_division=0))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, holdout_churn_prediction, display_labels=["Retained", "Churned"],
    ax=axes[0], colorbar=False
)
RocCurveDisplay.from_predictions(y_test, holdout_churn_probability, ax=axes[1])
axes[0].set_title("Holdout Confusion Matrix")
axes[1].set_title("Holdout ROC Curve")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "holdout_model_evaluation.png", dpi=160, bbox_inches="tight")
plt.show()
pd.DataFrame([holdout_metrics]).to_csv(OUTPUT_DIR / "holdout_metrics.csv", index=False)


## 10. Export the trained pipeline and score unseen customers

Only load `.joblib` files from trusted sources. This notebook saves the fitted preprocessing and classifier together, avoiding inconsistent transformations at inference time.

In [ ]:
model_path = MODEL_DIR / "telecom_customer_churn_pipeline.joblib"
joblib.dump(best_churn_pipeline, model_path)

# Reload the exact pipeline to validate the serialized inference workflow.
reloaded_churn_pipeline = joblib.load(model_path)
assert np.allclose(
    best_churn_pipeline.predict_proba(X_test),
    reloaded_churn_pipeline.predict_proba(X_test),
)

prediction_churn_probability = reloaded_churn_pipeline.predict_proba(prediction_features)[:, 1]
customer_churn_predictions = pd.DataFrame({
    "predicted_churn": np.where(prediction_churn_probability >= 0.5, "Yes", "No"),
    "churn_probability": prediction_churn_probability,
})
if CUSTOMER_ID_COLUMN in prediction_customers:
    customer_churn_predictions.insert(0, CUSTOMER_ID_COLUMN,
                                      prediction_customers[CUSTOMER_ID_COLUMN].values)

predictions_path = OUTPUT_DIR / "unseen_customer_churn_predictions.csv"
customer_churn_predictions.to_csv(predictions_path, index=False)
print("Saved model:", model_path)
print("Saved predictions:", predictions_path)
display(customer_churn_predictions.head())


## 11. Business interpretation and limitations

- **Review observed segment churn rates** for contract type, internet service, payment method, tenure, and monthly charges; prioritize segments using both churn rate and customer volume.
- **Investigate potential retention actions** (service quality checks, onboarding, plan-fit reviews) before assuming that observed associations are causal.
- **Do not treat churn scores as calibrated probabilities** without assessing calibration on a separate validation sample.
- **Fairness and privacy:** inspect performance across relevant customer groups, restrict access to customer-level outputs, and avoid discriminatory targeting.
- **Reproducibility:** the `outputs/` folder contains plots and model metrics; the `models/` folder contains the fitted pipeline. Neither directory should contain real customer records in a public Git repository.
- **Evaluation note:** reported metrics are generated when the notebook is executed with the actual data; no scores are claimed in advance.

**Suggested next steps:** threshold tuning using business costs, calibration, temporal validation if dates become available, and explainability for the final trained model.